# Tail Risk Horse Race v2, Bayesian LSTM-AL + CAViaR-AS

**Authors**: Jimena Huillca, Liqi Yi  (BSE Master in Financial Economics, 2026)  
**Advisor**: Prof. Argimiro Arratia Quesada

## Research Question
> *Does CETI improve prediction of bank equity tail risk (VaR / ES)?*

## What changed vs. v1
The MAP-optimised LSTM is replaced by the **Bayesian LSTM-AL** with adaptive MCMC
(Roberts & Rosenthal 2009). MAP optimisation was found to be unreliable for this
architecture, the non-convex loss landscape contains flat regions where the
optimizer produces near-zero violation rates. MCMC explores the full posterior and
produces well-calibrated forecasts from the posterior mean parameter vector.

## Four Models
| Model | Family | Inputs | Method |
|--|--|--|--|
| `CAViaR_BASE` | CAViaR-AS | $r_{t-1}$ | Quantile regression |
| `CAViaR_CETI` | CAViaR-AS | $r_{t-1}$, CETI$_{t-1}$ | Quantile regression |
| `LSTM_BASE`   | LSTM-AL   | $r_{t-1}$ | Bayesian MCMC |
| `LSTM_CETI`   | LSTM-AL   | $r_{t-1}$, CETI$_{t-1}$ | Bayesian MCMC |

**CETI window**: W=104 weeks throughout -- consistent with CLRI_Analysis pipeline.

## Run Order
1. **Cells 1–3**, imports, config, data (< 1 min)  
2. **Cell 4**, CAViaR functions (define only)  
3. **Cell 5**, LSTM-MCMC functions (define only)  
4. **Cell 6**, Evaluation functions (define only)  
5. **Cell 7**, ▶ Run CAViaR (~5 min)  
6. **Cell 8**, ▶ Run LSTM-MCMC (~25–35 min) ← run and wait  
7. **Cells 9–12**, DM tests, tables, plots, conclusion


## Cell 1, Imports & Configuration


In [20]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import chi2, norm
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
BASE       = '.'  # repo root
INPUT_PATH = 'clri_lstm_ready.csv'
OUTPUT_DIR = os.path.join('.', 'outputs', 'horserace')
PLOTS_DIR  = os.path.join(OUTPUT_DIR, 'PLOTS')
TABS_DIR   = os.path.join(OUTPUT_DIR, 'TABLES')
MCMC_DIR   = 'OUTPUT_HORSERACE_V2/MCMC_CHAINS'  # local path; GitHub users: chains are at repo root
for _d in [OUTPUT_DIR, PLOTS_DIR, TABS_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
ASSETS        = ['BBVA', 'BCP']
ALPHAS        = [0.05, 0.01]
CUTOFF        = pd.Timestamp('2023-03-10')
R_SCALE       = 100.0   # multiply log-returns by 100 → % units

RETURN_COLS = {'BBVA': 'dp_BBVA',        'BCP': 'dp_BCP'}
CETI_COLS   = {'BBVA': 'CETI_BBVA_W104', 'BCP': 'CETI_BCP_W104'}

# MCMC settings  (N_ITER=10000 takes ~8-10 min per variant on Mac M-series)
N_ITER         = 10_000
BURN_IN        = 3_000
ADAPT_START    = 1_000
ADAPT_INTERVAL = 200

RNG = np.random.default_rng(20260520)

CRISIS_SHADING = [
    ('2008-09-01', '2009-06-30', 'GFC'),
    ('2011-07-01', '2012-06-30', 'Euro Debt'),
    ('2015-08-01', '2016-03-31', 'China/EM'),
    ('2020-02-01', '2020-09-30', 'COVID-19'),
    ('2021-04-01', '2021-12-31', 'Peru Political'),
    ('2022-01-01', '2022-12-31', 'Rate Hike'),
]

print('Configuration loaded.')
print(f'MCMC: {N_ITER} iterations, {BURN_IN} burn-in → {N_ITER-BURN_IN} posterior draws')


Configuration loaded.
MCMC: 10000 iterations, 3000 burn-in → 7000 posterior draws


## Cell 2, Data Loading

Builds per-asset aligned datasets. All models share the same observations
(dropping rows where CETI_W104 is NaN ensures this).

**Lag convention**: CETI_{t-1} and r_{t-1} both lagged by 1 period -- no look-ahead.


In [21]:
def load_data():
    df = pd.read_csv(INPUT_PATH, parse_dates=['fecha'])
    df = df.set_index('fecha').sort_index()
    
    # Rename legacy CLRI columns to CETI for consistency with current terminology
    df = df.rename(columns={
        'CLRI_BBVA_W104': 'CETI_BBVA_W104',
        'CLRI_BCP_W104':  'CETI_BCP_W104',
        'CLRI_BBVA_W52':  'CETI_BBVA_W52',
        'CLRI_BCP_W52':   'CETI_BCP_W52',
        'CLRI_SYS_AW_W104': 'CETI_SYS_AW_W104',
        'CLRI_SYS_AW_W52':  'CETI_SYS_AW_W52',
        'CLRI_SYS_EW_W52':  'CETI_SYS_EW_W52',
    })
    
    print(f'Raw data: {df.shape[0]} weeks  {df.index.min().date()} -- {df.index.max().date()}')
    datasets = {}
    for asset in ASSETS:
        data = pd.DataFrame({
            'y':        df[RETURN_COLS[asset]],
            'r_lag':    df[RETURN_COLS[asset]].shift(1),
            'ceti_lag': df[CETI_COLS[asset]].shift(1),
            'crisis':   df['D_CRISIS'],
        }).dropna().copy()
        data['sample'] = np.where(data.index <= CUTOFF, 'train', 'test')
        datasets[asset] = data
        tr = data[data['sample']=='train']
        te = data[data['sample']=='test']
        print(f'  {asset}: train={len(tr)}  test={len(te)}  crisis_test={te["crisis"].sum():.0f}')
    return datasets

DATASETS = load_data()

Raw data: 1009 weeks  2007-01-12 -- 2026-05-08
  BBVA: train=480  test=139  crisis_test=16
  BCP: train=479  test=139  crisis_test=16


## Cell 3, CAViaR Functions (Asymmetric Slope)

**BASE** (4 params): $Q_t = \beta_0 + \beta_1 Q_{t-1} + \beta_2 r^+_{t-1} + \beta_3 r^-_{t-1}$  
**CETI** (5 params): same $+ \beta_4 \text{CETI}_{t-1}$

Estimated by minimising quantile (pinball) loss via L-BFGS-B + Powell,
80 random starts + 5 deterministic starts.


In [22]:
def pinball_loss(y, q, alpha):
    hit = (y < q).astype(float)
    return float(np.mean((alpha - hit) * (y - q)))

def pinball_per_period(y, q, alpha):
    hit = (y < q).astype(float)
    return (alpha - hit) * (y - q)

def caviar_recursion(params, y, rp, rn, cl, var_init):
    T = len(y); v = np.zeros(T); v[0] = var_init
    b0, b1, b2, b3 = params[:4]
    b4 = params[4] if len(params) == 5 else 0.0
    for t in range(1, T):
        v[t] = b0 + b1*v[t-1] + b2*rp[t-1] + b3*rn[t-1] + b4*cl[t-1]
    return v

def caviar_obj(p, y, rp, rn, cl, alpha, vi):
    v = caviar_recursion(p, y, rp, rn, cl, vi)
    return pinball_loss(y[1:], v[1:], alpha)

def caviar_bounds(use_ceti):                          # was use_clri
    b = [(-20., 0.), (0., 0.98), (-20., 20.), (0., 20.)]
    if use_ceti: b.append((-500., 500.))              # was use_clri
    return b

def caviar_starts(y_pct, alpha, use_ceti, n=80):      # was use_clri
    q = np.quantile(y_pct, alpha); vol = np.std(y_pct)
    det = [[q*.15,.5,-.1,.1],[q*.25,.7,-.2,.2],[q*.4,.85,-.3,.3],
           [-0.1*vol,.6,-.1,.1],[-0.2*vol,.8,-.2,.2]]
    if use_ceti: det = [s+[0.] for s in det]          # was use_clri
    bnd = caviar_bounds(use_ceti)                      # was use_clri
    rand = [[RNG.uniform(lo,hi) for lo,hi in bnd] for _ in range(n)]
    return det + rand

def fit_caviar(d_tr, alpha, use_ceti):
    y  = d_tr['y'].values * R_SCALE
    rp = np.maximum(d_tr['r_lag'].values * R_SCALE, 0.)
    rn = np.minimum(d_tr['r_lag'].values * R_SCALE, 0.)
    cl = d_tr['ceti_lag'].values
    vi = float(np.quantile(y, alpha))
    best_l, best_p = np.inf, None
    for x0 in caviar_starts(y, alpha, use_ceti):
        for m in ['L-BFGS-B','Powell']:
            try:
                r = minimize(caviar_obj, np.array(x0,float),
                             args=(y,rp,rn,cl,alpha,vi), method=m,
                             bounds=caviar_bounds(use_ceti),  # was use_clri
                             options={'maxiter':4000,'disp':False})
                if np.isfinite(r.fun) and r.fun < best_l:
                    best_l, best_p = r.fun, r.x
            except: pass
    return best_p, vi

def es_ratio(y_tr, v_tr, alpha):
    mask = y_tr < v_tr
    if mask.sum() >= 3:
        mr, mv = y_tr[mask].mean(), v_tr[mask].mean()
        if mv != 0: return float(np.clip(mr/mv, 1., 5.))
    return norm.pdf(norm.ppf(alpha)) / alpha

def caviar_forecast(params, d_tr, d_te, alpha, vi):
    """Full (train+test) recursion to carry hidden state into test."""
    dall = pd.concat([d_tr, d_te])
    y    = dall['y'].values * R_SCALE
    rp   = np.maximum(dall['r_lag'].values * R_SCALE, 0.)
    rn   = np.minimum(dall['r_lag'].values * R_SCALE, 0.)
    cl   = dall['ceti_lag'].values                # was clri_lag
    v    = caviar_recursion(params, y, rp, rn, cl, vi)
    n    = len(d_tr)
    c    = es_ratio(y[:n], v[:n], alpha)
    return v[n:], c * v[n:]  # var_test_pct, es_test_pct

print('CAViaR functions defined.')


CAViaR functions defined.


## Cell 4, LSTM-AL Functions (Bayesian MCMC)

Architecture: single-layer LSTM (scalar hidden state) with hybrid
CAViaR-style quantile output. Joint VaR + ES estimation via the
Asymmetric Laplace quasi-likelihood (Fissler & Ziegel 2016).

**LSTM cell** (scalar $h_t$, standard 4-gate structure):
$$f_t=\sigma(W_f x_t+U_f h_{t-1}+b_f),\quad i_t,o_t,g_t \text{ analogously}$$
$$c_t=f_t c_{t-1}+i_t g_t,\quad h_t=o_t\tanh(c_t)$$

**Output**:
$$Q_t(\tau)=a_0+a_1 h_t+b_0|r_{t-1}|+b_1 Q_{t-1},\quad
\text{ES}_t=(1+\text{softplus}(g_0+g_1 h_t))Q_t$$

**Parameter count**: BASE=18, CETI=22.

**Estimation**: MAP initialisation (Powell, 9 starts) → 10 000-iteration
adaptive MCMC → posterior mean used for point forecasts.


In [23]:
# ── Numerically stable activations ───────────────────────────────────────
def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))

def softplus(x):
    x = np.clip(x, -40, 40)
    return np.log1p(np.exp(x))

def inv_softplus(y):
    y = max(float(y), 1e-8)
    return np.log(np.exp(y) - 1.0)


# ── Parameter layout ─────────────────────────────────────────────────────
def lstm_n_params(input_dim):
    return 4 * (input_dim + 2) + 6


def lstm_split(theta, input_dim):
    idx, gates = 0, {}
    for gate in ['f', 'i', 'o', 'c']:
        W = theta[idx:idx+input_dim]; idx += input_dim
        U = theta[idx]; idx += 1
        b = theta[idx]; idx += 1
        gates[gate] = {'W': W, 'U': U, 'b': b}
    ra0, ra1, rb0, rb1, rg0, rg1 = theta[idx:idx+6]
    out = {
        'a0': -softplus(ra0),
        'a1':  0.5 * np.tanh(ra1),
        'b0': -softplus(rb0),
        'b1':  0.98 * sigmoid(rb1),
        'g0':  rg0,
        'g1':  0.5 * np.tanh(rg1),
    }
    return gates, out


# ── LSTM forward pass ────────────────────────────────────────────────────
def lstm_forward(theta, X, r_lag_pct, y_pct, alpha):
    """
    One-step-ahead VaR and ES sequence.
    X         : standardised features (T, input_dim)
    r_lag_pct : lagged returns in %  (T,) , for |r_{t-1}| in VaR output
    y_pct     : target returns in %  (T,) , only to initialise VaR[0]
    """
    T, input_dim = X.shape
    gates, out = lstm_split(theta, input_dim)
    h = c = 0.0
    VaR = np.zeros(T); ES = np.zeros(T)
    VaR[0] = np.quantile(y_pct, alpha)
    tail   = y_pct[y_pct <= VaR[0]]
    ES[0]  = tail.mean() if len(tail) > 0 else 1.2 * VaR[0]
    if ES[0] >= VaR[0]: ES[0] = 1.2 * VaR[0]

    for t in range(1, T):
        x = X[t]
        f = sigmoid(np.dot(gates['f']['W'], x) + gates['f']['U']*h + gates['f']['b'])
        i = sigmoid(np.dot(gates['i']['W'], x) + gates['i']['U']*h + gates['i']['b'])
        o = sigmoid(np.dot(gates['o']['W'], x) + gates['o']['U']*h + gates['o']['b'])
        g = np.tanh( np.dot(gates['c']['W'], x) + gates['c']['U']*h + gates['c']['b'])
        c = f*c + i*g
        h = o * np.tanh(c)

        q = out['a0'] + out['a1']*h + out['b0']*abs(r_lag_pct[t]) + out['b1']*VaR[t-1]
        e = (1.0 + softplus(out['g0'] + out['g1']*h)) * q
        VaR[t] = q; ES[t] = e
    return VaR, ES


# ── AL quasi-likelihood ───────────────────────────────────────────────────
def al_log_likelihood(y_pct, VaR_pct, ES_pct, alpha):
    """
    Fissler & Ziegel (2016) joint VaR/ES log-likelihood (AL loss).
    Returns -inf for invalid (VaR, ES) pairs.
    """
    y = y_pct[1:]; V = VaR_pct[1:]; E = ES_pct[1:]
    if not (np.all(np.isfinite(V)) and np.all(np.isfinite(E))): return -np.inf
    if np.any(V >= -1e-8) or np.any(E >= V): return -np.inf
    ratio = (alpha - 1.0) / E
    if np.any(ratio <= 0): return -np.inf
    hit = (y <= V).astype(float)
    loss = -np.log(ratio) - (y - V) * (alpha - hit) / (alpha * E)
    if not np.all(np.isfinite(loss)): return -np.inf
    return -np.sum(loss)   # return neg-loss = log-likelihood


def log_prior(theta):
    """Weakly informative Gaussian prior (sd=2.5), regularises extreme draws."""
    return -0.5 * np.sum((theta / 2.5) ** 2)


def log_posterior(theta, X, r_lag_pct, y_pct, alpha):
    VaR, ES = lstm_forward(theta, X, r_lag_pct, y_pct, alpha)
    ll = al_log_likelihood(y_pct, VaR, ES, alpha)
    if not np.isfinite(ll): return -np.inf
    return ll + log_prior(theta)


# ── MAP initialisation ────────────────────────────────────────────────────
def initial_theta(input_dim, y_pct, alpha):
    """Economically calibrated starting vector (steady-state VaR ≈ empirical quantile)."""
    n = lstm_n_params(input_dim)
    theta = np.zeros(n)
    q_a = np.quantile(y_pct, alpha)   # negative
    b1_init = 0.60
    a0_abs  = abs(q_a * (1.0 - b1_init))
    theta[-6] = inv_softplus(a0_abs)
    theta[-5] = 0.0
    theta[-4] = inv_softplus(0.05)
    theta[-3] = np.log(b1_init / (0.98 - b1_init))
    theta[-2] = -1.0
    theta[-1] = 0.0
    return theta


def find_map(X_tr, r_lag_tr, y_tr, alpha, input_dim):
    """MAP via Powell optimiser (9 starts). Returns best_theta."""
    base   = initial_theta(input_dim, y_tr, alpha)
    starts = [base] + [base + RNG.normal(0, 0.05, len(base)) for _ in range(8)]
    best   = None
    for s in starts:
        try:
            r = minimize(
                lambda t: -log_posterior(t, X_tr, r_lag_tr, y_tr, alpha),
                x0=s, method='Powell',
                options={'maxiter': 4000, 'disp': False, 'xtol': 1e-5, 'ftol': 1e-5})
            if best is None or r.fun < best.fun: best = r
        except: pass
    return best.x if best else base


# ── Adaptive MCMC sampler ─────────────────────────────────────────────────
def run_mcmc(theta0, X_tr, r_lag_tr, y_tr, alpha,
             n_iter=N_ITER, burn_in=BURN_IN):
    """
    Adaptive random-walk Metropolis-Hastings (Roberts & Rosenthal 2009).
    Returns (posterior_samples, acceptance_rate).
    """
    dim = len(theta0)
    samples_all = np.zeros((n_iter, dim))
    theta_curr  = theta0.copy()
    lp_curr     = log_posterior(theta_curr, X_tr, r_lag_tr, y_tr, alpha)
    prop_cov    = (0.01**2) * np.eye(dim)
    accepted    = 0

    for it in range(n_iter):
        theta_prop = RNG.multivariate_normal(theta_curr, prop_cov)
        lp_prop    = log_posterior(theta_prop, X_tr, r_lag_tr, y_tr, alpha)
        if np.isfinite(lp_prop) and np.log(RNG.uniform()) < lp_prop - lp_curr:
            theta_curr, lp_curr = theta_prop, lp_prop
            accepted += 1
        samples_all[it] = theta_curr

        # Adapt proposal covariance
        if it >= ADAPT_START and it % ADAPT_INTERVAL == 0:
            emp = np.cov(samples_all[:it+1].T)
            prop_cov = (2.38**2 / dim) * emp + 1e-7 * np.eye(dim)

    return samples_all[burn_in:], accepted / n_iter


# ── Build feature matrix ──────────────────────────────────────────────────
def build_lstm_arrays(d_tr, d_te, use_ceti):          # was use_clri
    """
    Standardise inputs using training moments only.
    Returns X_train, X_test, r_lag_train_pct, r_lag_test_pct, y_tr_pct, y_te_pct.
    """
    r_tr = d_tr['r_lag'].values * R_SCALE
    r_te = d_te['r_lag'].values * R_SCALE
    y_tr = d_tr['y'].values * R_SCALE
    y_te = d_te['y'].values * R_SCALE

    if use_ceti:                                       # was use_clri
        cl_tr = d_tr['ceti_lag'].values               # was clri_lag
        cl_te = d_te['ceti_lag'].values               # was clri_lag
        raw_tr = np.column_stack([r_tr, cl_tr])
        raw_te = np.column_stack([r_te, cl_te])
    else:
        raw_tr = r_tr.reshape(-1, 1)
        raw_te = r_te.reshape(-1, 1)

    # Standardise using training stats only
    mu  = raw_tr.mean(axis=0)
    sig = raw_tr.std(axis=0)
    sig = np.where(sig > 1e-8, sig, 1.0)
    X_tr = (raw_tr - mu) / sig
    X_te = (raw_te - mu) / sig
    return X_tr, X_te, r_tr, r_te, y_tr, y_te


# ── Posterior forecast ────────────────────────────────────────────────────
def lstm_posterior_forecast(samples, d_tr, d_te, alpha, use_ceti):   # was use_clri
    """
    Forecast with posterior-mean parameters.
    Runs the full train+test sequence so LSTM state carries forward cleanly.
    Returns var_test_pct, es_test_pct (in % units).
    """
    theta_mean = samples.mean(axis=0)
    X_tr, X_te, r_tr, r_te, y_tr, y_te = build_lstm_arrays(d_tr, d_te, use_ceti)  # was use_clri

    X_all     = np.vstack([X_tr, X_te])
    r_lag_all = np.concatenate([r_tr, r_te])
    y_all     = np.concatenate([y_tr, y_te])

    VaR_all, ES_all = lstm_forward(theta_mean, X_all, r_lag_all, y_all, alpha)
    n = len(y_tr)
    return VaR_all[n:], ES_all[n:]

print('LSTM-MCMC functions defined.')
print(f'Parameter counts: LSTM_BASE={lstm_n_params(1)}, LSTM_CETI={lstm_n_params(2)}')


LSTM-MCMC functions defined.
Parameter counts: LSTM_BASE=18, LSTM_CETI=22


## Cell 5, Evaluation Functions

Implements Kupiec (1995), Christoffersen (1998), Acerbi-Szekely Z₂,
Diebold-Mariano (1995), and the per-model scorecard function.


In [24]:
def kupiec(y, v, alpha):
    hits = (y < v).astype(int); n, T = hits.sum(), len(hits)
    if n == 0 or n == T: return np.nan, np.nan, n/T, n
    p = n/T
    lr = -2.*(n*np.log(alpha/p) + (T-n)*np.log((1-alpha)/(1-p)))
    return round(float(lr),3), round(float(chi2.sf(lr,1)),4), round(p,4), int(n)

def christoffersen(y, v, alpha):
    hits = (y < v).astype(int); T = len(hits); n = hits.sum()
    if n == 0 or n == T or T < 2: return np.nan, np.nan, np.nan, np.nan
    n00=n01=n10=n11=0
    for t in range(1,T):
        p,c = hits[t-1], hits[t]
        if p==0 and c==0: n00+=1
        elif p==0 and c==1: n01+=1
        elif p==1 and c==0: n10+=1
        else: n11+=1
    pi01 = n01/(n00+n01) if (n00+n01)>0 else 0.
    pi11 = n11/(n10+n11) if (n10+n11)>0 else 0.
    pi   = n/T
    sl   = lambda x: np.log(max(x,1e-300))
    lr_i = -2.*((n00+n01)*sl(1-pi)+n*sl(pi)-n00*sl(1-pi01)-n01*sl(pi01+1e-300)
                -n10*sl(1-pi11)-n11*sl(pi11+1e-300))
    lr_u = kupiec(y,v,alpha)[0]
    if not np.isfinite(lr_u): return np.nan,np.nan,np.nan,np.nan
    lr_cc = lr_u + lr_i
    return round(float(lr_cc),3), round(float(chi2.sf(max(lr_cc,0),2)),4), \
           round(float(lr_i),3),  round(float(chi2.sf(max(lr_i,0),1)),4)

def z2_test(y, v, e, alpha):
    T = len(y); hits = (y<v).astype(float); nv = hits.sum()
    if nv == 0: return np.nan, np.nan
    with np.errstate(divide='ignore',invalid='ignore'):
        ratio = np.where(np.abs(e)>1e-8, y/e, 0.)
    z2  = float(np.sum(ratio*hits)/(T*alpha) + 1.)
    sig = float(np.sqrt(np.var(ratio[hits==1])/nv)) if nv>1 else 1.
    pv  = float(norm.cdf(z2/(sig+1e-8))) if sig>0 else np.nan
    return round(z2,4), round(pv,4)

def dm_test(y, v1, v2, alpha):
    """Positive DM → model2 is better (lower pinball loss)."""
    h1=(y<v1).astype(float); h2=(y<v2).astype(float)
    l1=(alpha-h1)*(y-v1); l2=(alpha-h2)*(y-v2); d=l1-l2; T=len(d)
    d_bar = d.mean()
    g0 = np.mean((d-d_bar)**2)
    nw = g0 + 2*(1-1/2)*np.mean((d[1:]-d_bar)*(d[:-1]-d_bar))
    if nw <= 0: return np.nan, np.nan, round(float(d_bar),5)
    dm = float(d_bar/np.sqrt(nw/T))
    return round(dm,3), round(float(2*norm.sf(abs(dm))),4), round(float(d_bar),5)

def scorecard(name, asset, alpha, y_te, v_pct, e_pct, crisis_mask):
    lr_u,p_u,vr,nv = kupiec(y_te,v_pct,alpha)
    lr_cc,p_cc,lr_i,p_i = christoffersen(y_te,v_pct,alpha)
    pb  = round(pinball_loss(y_te,v_pct,alpha),5)
    z2,pz2 = z2_test(y_te,v_pct,e_pct,alpha)
    if crisis_mask.sum()>0:
        yc,vc,ec = y_te[crisis_mask],v_pct[crisis_mask],e_pct[crisis_mask]
        vrc = round(float((yc<vc).mean()),4)
        pbc = round(pinball_loss(yc,vc,alpha),5)
        z2c,_ = z2_test(yc,vc,ec,alpha)
    else:
        vrc=pbc=z2c=np.nan
    return {'Model':name,'Asset':asset,'Alpha':alpha,'N_test':len(y_te),
            'N_viol':nv,'ViolRate':vr,'Kupiec_p':p_u,'CC_p':p_cc,'Ind_p':p_i,
            'Pinball':pb,'Z2':z2,'Z2_p':pz2,
            'ViolRate_crisis':vrc,'Pinball_crisis':pbc,'Z2_crisis':z2c}

print('Evaluation functions defined.')


Evaluation functions defined.


## Cell 6, Run CAViaR (≈ 5 min)

Estimates CAViaR_BASE and CAViaR_CETI for all (asset, alpha) combinations.
Saves VaR/ES forecast series for downstream evaluation.


In [25]:
RESULTS   = []
FORECASTS = {}

def sep(msg): print('\n'+'═'*60+'\n '+msg+'\n'+'═'*60)

sep('CAViaR Estimation')
for asset in ASSETS:
    d = DATASETS[asset]
    d_tr = d[d['sample']=='train']
    d_te = d[d['sample']=='test']
    y_te = d_te['y'].values * R_SCALE
    cm   = d_te['crisis'].values.astype(bool)

    for alpha in ALPHAS:
        print(f'\n  {asset}  α={alpha:.0%}')
        for use_ceti, mname in [(False,'CAViaR_BASE'),(True,'CAViaR_CETI')]:
            t0 = time.time()
            params, vi = fit_caviar(d_tr, alpha, use_ceti)
            v_te, e_te = caviar_forecast(params, d_tr, d_te, alpha, vi)
            sc = scorecard(mname, asset, alpha, y_te, v_te, e_te, cm)
            RESULTS.append(sc)
            FORECASTS[(mname, asset, alpha)] = {'var': v_te, 'es': e_te}
            b4 = f'  β₄={params[4]:.2f}' if use_ceti else ''
            print(f'  {mname}: viol={sc["ViolRate"]:.1%}  Kupiec_p={sc["Kupiec_p"]}  '
                  f'Pinball={sc["Pinball"]}  [{time.time()-t0:.0f}s]{b4}')

print('\n✓ CAViaR done.')


════════════════════════════════════════════════════════════
 CAViaR Estimation
════════════════════════════════════════════════════════════

  BBVA  α=5%
  CAViaR_BASE: viol=1.4%  Kupiec_p=0.0239  Pinball=0.11555  [16s]
  CAViaR_CETI: viol=1.4%  Kupiec_p=0.0239  Pinball=0.11499  [23s]  β₄=8.72

  BBVA  α=1%
  CAViaR_BASE: viol=0.7%  Kupiec_p=0.7263  Pinball=0.03819  [17s]
  CAViaR_CETI: viol=1.4%  Kupiec_p=0.6256  Pinball=0.03786  [23s]  β₄=13.72

  BCP  α=5%
  CAViaR_BASE: viol=3.6%  Kupiec_p=0.4253  Pinball=0.27587  [16s]
  CAViaR_CETI: viol=5.0%  Kupiec_p=0.9845  Pinball=0.26856  [22s]  β₄=49.34

  BCP  α=1%
  CAViaR_BASE: viol=1.4%  Kupiec_p=0.6256  Pinball=0.08211  [17s]
  CAViaR_CETI: viol=2.9%  Kupiec_p=0.0699  Pinball=0.10006  [23s]  β₄=85.98

✓ CAViaR done.


## Cell 7, Run LSTM-MCMC (≈ 25–35 min)

**Run this cell and wait**, it runs 8 MCMC chains
(2 assets × 2 variants × 2 α levels), each 10 000 iterations.

Progress is printed after each chain completes.
MCMC chains are saved to disk so you do not need to re-run if the kernel restarts.

**Interpretation note**: The posterior mean of $\theta$ is used for point forecasts.
No weight interpretability is claimed -- CETI is useful if it improves
out-of-sample VaR/ES accuracy.


In [26]:
import os as _os

sep('LSTM-MCMC Estimation')

LSTM_VARIANTS = {'LSTM_BASE': False, 'LSTM_CETI': True}

for asset in ASSETS:
    d    = DATASETS[asset]
    d_tr = d[d['sample']=='train']
    d_te = d[d['sample']=='test']
    y_te = d_te['y'].values * R_SCALE
    cm   = d_te['crisis'].values.astype(bool)

    for alpha in ALPHAS:
        for mname, use_ceti in LSTM_VARIANTS.items():   # was use_clri
            chain_file = _os.path.join(MCMC_DIR,
                f'chain_{mname}_{asset}_a{int(alpha*100):02d}.npy')

            if _os.path.exists(chain_file):
                samples = np.load(chain_file)
                print(f'  {mname} {asset} α={alpha:.0%}: loaded from cache'
                      f'  ({len(samples)} draws)')
            else:
                X_tr, _, r_tr, _, y_tr, _ = build_lstm_arrays(
                    d_tr, d_te, use_ceti)                # was use_clri
                input_dim = X_tr.shape[1]

                t0 = time.time()
                theta_map = find_map(X_tr, r_tr, y_tr, alpha, input_dim)
                t_map = time.time() - t0

                samples, acc_rate = run_mcmc(
                    theta_map, X_tr, r_tr, y_tr, alpha)
                t_total = time.time() - t0

                np.save(chain_file, samples)
                print(f'  {mname} {asset} α={alpha:.0%}: MAP={t_map:.0f}s  '
                      f'MCMC={t_total-t_map:.0f}s  '
                      f'accept={acc_rate:.1%}  saved')

            v_te, e_te = lstm_posterior_forecast(samples, d_tr, d_te,
                                                  alpha, use_ceti)  # was use_clri
            sc = scorecard(mname, asset, alpha, y_te, v_te, e_te, cm)
            RESULTS.append(sc)
            FORECASTS[(mname, asset, alpha)] = {'var': v_te, 'es': e_te}
            print(f'    → viol={sc["ViolRate"]:.1%}  Kupiec_p={sc["Kupiec_p"]}  '
                  f'Pinball={sc["Pinball"]}')

RESULTS_DF = pd.DataFrame(RESULTS)
print('\n✓ LSTM-MCMC done.  Total results:', len(RESULTS_DF))


════════════════════════════════════════════════════════════
 LSTM-MCMC Estimation
════════════════════════════════════════════════════════════
  LSTM_BASE BBVA α=5%: loaded from cache  (7000 draws)
    → viol=2.9%  Kupiec_p=0.2138  Pinball=0.10997
  LSTM_CETI BBVA α=5%: loaded from cache  (7000 draws)
    → viol=3.6%  Kupiec_p=0.4253  Pinball=0.11072
  LSTM_BASE BBVA α=1%: loaded from cache  (7000 draws)
    → viol=1.4%  Kupiec_p=0.6256  Pinball=0.03882
  LSTM_CETI BBVA α=1%: loaded from cache  (7000 draws)
    → viol=1.4%  Kupiec_p=0.6256  Pinball=0.04118
  LSTM_BASE BCP α=5%: loaded from cache  (7000 draws)
    → viol=2.9%  Kupiec_p=0.2138  Pinball=0.24782
  LSTM_CETI BCP α=5%: loaded from cache  (7000 draws)
    → viol=3.6%  Kupiec_p=0.4253  Pinball=0.26243
  LSTM_BASE BCP α=1%: loaded from cache  (7000 draws)
    → viol=0.7%  Kupiec_p=0.7263  Pinball=0.06197
  LSTM_CETI BCP α=1%: loaded from cache  (7000 draws)
    → viol=2.2%  Kupiec_p=0.2343  Pinball=0.06114

✓ LSTM-MCMC done. 

## Cell 7b -- Two-Stage LSTM-CETI  (`LSTM_CETI_2S`)

**Motivation**: The raw `LSTM_CETI` feeds CETI into a 22-parameter nonlinear
system, which can learn the wrong direction of effect in a small sample.
The two-stage model separates concerns cleanly:

$$\widehat{\text{VaR}}^{\text{CETI}}_t = a + b\,\widehat{\text{VaR}}^{\text{BASE}}_t + c\,\text{CETI}_{t-1}$$

where $a, b, c$ are estimated by minimising pinball loss on the **training set**
only, then applied to the test set without look-ahead.
The coefficient $c$ is the direct incremental-information test for CETI --
exactly analogous to $\beta_4$ in `CAViaR_CETI`.

**ES**: scaled proportionally to the VaR adjustment, preserving the
LSTM output-layer VaR/ES ratio.

**Run time**: < 5 s (no MCMC).


In [27]:
def lstm_2stage_forecast(asset, alpha, d_tr, d_te):
    """
    Stage 1: posterior-mean LSTM_BASE -> VaR_BASE on train + test.
    Stage 2: VaR_CETI_t = a + b*VaR_BASE_t + c*CETI_{t-1}
             estimated on training data via pinball-loss minimisation.
    ES scaled proportionally to preserve the LSTM VaR/ES ratio.
    """
    chain_file = _os.path.join(MCMC_DIR,
        f'chain_LSTM_BASE_{asset}_a{int(alpha*100):02d}.npy')
    samples    = np.load(chain_file)
    theta_mean = samples.mean(axis=0)

    # Full-sequence arrays (BASE = returns only)
    X_tr, X_te, r_tr, r_te, y_tr, y_te = build_lstm_arrays(
        d_tr, d_te, use_ceti=False)               # was use_clri
    X_all     = np.vstack([X_tr, X_te])
    r_lag_all = np.concatenate([r_tr, r_te])
    y_all     = np.concatenate([y_tr, y_te])

    VaR_all, ES_all = lstm_forward(theta_mean, X_all, r_lag_all, y_all, alpha)
    n_tr        = len(y_tr)
    VaR_base_tr = VaR_all[:n_tr]
    VaR_base_te = VaR_all[n_tr:]
    ES_base_te  = ES_all[n_tr:]

    # Standardise CETI using training moments only (no look-ahead)
    cl_tr_raw = d_tr['ceti_lag'].values           # was clri_lag
    cl_te_raw = d_te['ceti_lag'].values           # was clri_lag
    cl_mu = cl_tr_raw.mean()
    cl_sd = cl_tr_raw.std(); cl_sd = max(cl_sd, 1e-8)
    ceti_tr = (cl_tr_raw - cl_mu) / cl_sd        # was clri_tr
    ceti_te = (cl_te_raw - cl_mu) / cl_sd        # was clri_te

    # Fit on training data
    def obj(params):
        a, b, c = params
        return pinball_loss(y_tr, a + b*VaR_base_tr + c*ceti_tr, alpha)  # was clri_tr

    starts = [[0., 1., 0.], [0., 1., .5], [0., 1., -.5],
              [0., .9, .2], [0., 1.1, -.2]]
    best = None
    for x0 in starts:
        try:
            r = minimize(obj, x0, method='L-BFGS-B',
                         options={'maxiter': 3000, 'ftol': 1e-12})
            if best is None or r.fun < best.fun: best = r
        except: pass
    a, b, c = best.x

    # Apply to test period
    VaR_2s = a + b * VaR_base_te + c * ceti_te   # was clri_te

    # ES: proportional scaling, preserves LSTM VaR/ES ratio
    ratio = np.where(np.abs(VaR_base_te) > 1e-8,
                     VaR_2s / VaR_base_te, 1.0)
    ES_2s = ES_base_te * ratio
    ES_2s = np.minimum(ES_2s, VaR_2s - 1e-6)

    return VaR_2s, ES_2s, a, b, c

sep('Two-Stage LSTM  (LSTM_CETI_2S)')
for asset in ASSETS:
    d    = DATASETS[asset]
    d_tr = d[d['sample']=='train']
    d_te = d[d['sample']=='test']
    y_te = d_te['y'].values * R_SCALE
    cm   = d_te['crisis'].values.astype(bool)
    for alpha in ALPHAS:
        v2s, e2s, a_, b_, c_ = lstm_2stage_forecast(asset, alpha, d_tr, d_te)
        sc = scorecard('LSTM_CETI_2S', asset, alpha, y_te, v2s, e2s, cm)
        RESULTS.append(sc)
        FORECASTS[('LSTM_CETI_2S', asset, alpha)] = {'var': v2s, 'es': e2s}
        print(f'  {asset} α={alpha:.0%}:  a={a_:.4f}  b={b_:.4f}  c={c_:.4f}  '
              f'viol={sc["ViolRate"]:.1%}  Kupiec_p={sc["Kupiec_p"]}  '
              f'Pinball={sc["Pinball"]}')

RESULTS_DF = pd.DataFrame(RESULTS)
print('\n✓ Two-stage done.  Total results:', len(RESULTS_DF))


════════════════════════════════════════════════════════════
 Two-Stage LSTM  (LSTM_CETI_2S)
════════════════════════════════════════════════════════════
  BBVA α=5%:  a=-0.6228  b=0.6825  c=0.2993  viol=3.6%  Kupiec_p=0.4253  Pinball=0.10603
  BBVA α=1%:  a=0.0960  b=1.0178  c=0.2480  viol=1.4%  Kupiec_p=0.6256  Pinball=0.03954
  BCP α=5%:  a=-0.3324  b=0.9435  c=0.1661  viol=2.9%  Kupiec_p=0.2138  Pinball=0.24395
  BCP α=1%:  a=0.0724  b=1.0565  c=0.0149  viol=0.7%  Kupiec_p=0.7263  Pinball=0.06236

✓ Two-stage done.  Total results: 20


## Cell 8 -- Diebold-Mariano Tests: BASE vs CETI

Tests whether CETI adds statistically significant predictive accuracy
within each model family.

- **Positive DM** -- CETI model has lower pinball loss  
- p < 0.10 → borderline significant; p < 0.05 → significant


In [28]:
DM_ROWS = []
for asset in ASSETS:
    y_te = DATASETS[asset][DATASETS[asset]['sample']=='test']['y'].values * R_SCALE
    for alpha in ALPHAS:
        for fam, base_m, ceti_m in [                          # was clri_m
                ('CAViaR',   'CAViaR_BASE', 'CAViaR_CETI'),
                ('LSTM',     'LSTM_BASE',   'LSTM_CETI'),
                ('LSTM_2S',  'LSTM_BASE',   'LSTM_CETI_2S')]:
            v_b = FORECASTS[(base_m, asset, alpha)]['var']
            v_c = FORECASTS[(ceti_m, asset, alpha)]['var']    # was clri_m
            dm, pdm, db = dm_test(y_te, v_b, v_c, alpha)
            DM_ROWS.append({'Family':fam,'Asset':asset,'Alpha':alpha,
                'DM_stat':dm,'p_value':pdm,'LossDiff':db,
                'Conclusion':'CETI better' if (dm>0 and pdm<.10) else
                             'BASE better' if (dm<0 and pdm<.10) else 'ns'})

DM_DF = pd.DataFrame(DM_ROWS)
print(DM_DF.to_string(index=False))
DM_DF.to_csv(_os.path.join(TABS_DIR,'dm_test_results.csv'), index=False)

 Family Asset  Alpha  DM_stat  p_value  LossDiff  Conclusion
 CAViaR  BBVA   0.05    0.811   0.4176   0.00056          ns
   LSTM  BBVA   0.05   -0.182   0.8554  -0.00076          ns
LSTM_2S  BBVA   0.05    2.965   0.0030   0.00393 CETI better
 CAViaR  BBVA   0.01    0.071   0.9432   0.00033          ns
   LSTM  BBVA   0.01   -0.308   0.7578  -0.00236          ns
LSTM_2S  BBVA   0.01   -0.340   0.7339  -0.00072          ns
 CAViaR   BCP   0.05    0.755   0.4503   0.00731          ns
   LSTM   BCP   0.05   -0.978   0.3281  -0.01460          ns
LSTM_2S   BCP   0.05    1.129   0.2590   0.00388          ns
 CAViaR   BCP   0.01   -1.238   0.2158  -0.01795          ns
   LSTM   BCP   0.01    0.182   0.8557   0.00083          ns
LSTM_2S   BCP   0.01   -0.243   0.8079  -0.00039          ns


## Cell 9, Horse Race Summary Tables

Primary results at τ=5%, robustness check at τ=1%.

**Coverage pass criterion**: Kupiec p > 0.05 (fail to reject $H_0$: coverage = τ).
**ES quality**: Z₂ closer to 0 is better; large negative = ES under-estimated.


In [29]:
cols = ['Model','Asset','Alpha','N_viol','ViolRate','Kupiec_p','CC_p',
        'Pinball','Z2','Pinball_crisis']

for tau in [0.05, 0.01]:
    sub = RESULTS_DF[RESULTS_DF['Alpha']==tau][cols].copy()
    sub = sub.sort_values(['Asset','Model'])
    lbl = '5%' if tau==0.05 else '1%'
    print(f'\n── τ = {lbl} ────────────────────────────────────────────────')
    print(sub.to_string(index=False))

RESULTS_DF.to_csv(_os.path.join(TABS_DIR,'horse_race_all.csv'), index=False)
print(f'\nSaved → {TABS_DIR}')



── τ = 5% ────────────────────────────────────────────────
       Model Asset  Alpha  N_viol  ViolRate  Kupiec_p   CC_p  Pinball     Z2  Pinball_crisis
 CAViaR_BASE  BBVA   0.05       2    0.0144    0.0239 0.0758  0.11555 1.3603         0.10365
 CAViaR_CETI  BBVA   0.05       2    0.0144    0.0239 0.0758  0.11499 1.3815         0.10835
   LSTM_BASE  BBVA   0.05       4    0.0288    0.2138 0.1038  0.10997 1.6901         0.09734
   LSTM_CETI  BBVA   0.05       5    0.0360    0.4253 0.0164  0.11072 1.8793         0.09801
LSTM_CETI_2S  BBVA   0.05       5    0.0360    0.4253 0.2572  0.10603 1.8212         0.09447
 CAViaR_BASE   BCP   0.05       5    0.0360    0.4253 0.6028  0.27587 1.8130         0.18497
 CAViaR_CETI   BCP   0.05       7    0.0504    0.9845 0.6331  0.26856 2.3219         0.18576
   LSTM_BASE   BCP   0.05       4    0.0288    0.2138 0.4096  0.24782 1.5055         0.19357
   LSTM_CETI   BCP   0.05       5    0.0360    0.4253 0.6028  0.26243 1.8068         0.14262
LSTM_CETI_

## Cell 10, Crisis-Period Deep Dive

Full / Calm / Crisis breakdown at τ=5%.
CETI's expected value-add: tighter VaR during crisis periods.


In [30]:
print('Crisis-period breakdown  (τ=5%)')
print('='*70)
CB = []
for asset in ASSETS:
    d   = DATASETS[asset]; d_te = d[d['sample']=='test']
    y   = d_te['y'].values * R_SCALE
    cri = d_te['crisis'].values.astype(bool); calm = ~cri
    alpha = 0.05
    print(f'\n{asset}  (crisis={cri.sum()}, calm={calm.sum()})')
    print(f'  {"Model":<18} {"Full":>8}{"Calm":>8}{"Crisis":>8}  │  '
          f'{"Full":>7}{"Calm":>7}{"Crisis":>7}  [Pinball / ViolRate]')
    for m in ['CAViaR_BASE','CAViaR_CETI','LSTM_BASE','LSTM_CETI','LSTM_CETI_2S']:
        v = FORECASTS[(m,asset,alpha)]['var']
        pb_f = pinball_loss(y,v,alpha)
        pb_c = pinball_loss(y[calm],v[calm],alpha)   if calm.sum()>0  else np.nan
        pb_k = pinball_loss(y[cri], v[cri], alpha)   if cri.sum()>0   else np.nan
        vr_f = float((y<v).mean())
        vr_c = float((y[calm]<v[calm]).mean())  if calm.sum()>0  else np.nan
        vr_k = float((y[cri]<v[cri]).mean())    if cri.sum()>0   else np.nan
        print(f'  {m:<18} {pb_f:>8.4f}{pb_c:>8.4f}{pb_k:>8.4f}  │  '
              f'{vr_f:>7.1%}{vr_c:>7.1%}{vr_k:>7.1%}')
        CB.append({'Model':m,'Asset':asset,'Alpha':alpha,
                   'Pinball_full':pb_f,'Pinball_calm':pb_c,'Pinball_crisis':pb_k,
                   'ViolRate_full':vr_f,'ViolRate_calm':vr_c,'ViolRate_crisis':vr_k})

pd.DataFrame(CB).to_csv(_os.path.join(TABS_DIR,'crisis_breakdown.csv'),index=False)


Crisis-period breakdown  (τ=5%)

BBVA  (crisis=16, calm=123)
  Model                  Full    Calm  Crisis  │     Full   Calm Crisis  [Pinball / ViolRate]
  CAViaR_BASE          0.1155  0.1171  0.1037  │     1.4%   1.6%   0.0%
  CAViaR_CETI          0.1150  0.1159  0.1084  │     1.4%   1.6%   0.0%
  LSTM_BASE            0.1100  0.1116  0.0973  │     2.9%   3.3%   0.0%
  LSTM_CETI            0.1107  0.1124  0.0980  │     3.6%   4.1%   0.0%
  LSTM_CETI_2S         0.1060  0.1075  0.0945  │     3.6%   4.1%   0.0%

BCP  (crisis=16, calm=123)
  Model                  Full    Calm  Crisis  │     Full   Calm Crisis  [Pinball / ViolRate]
  CAViaR_BASE          0.2759  0.2877  0.1850  │     3.6%   4.1%   0.0%
  CAViaR_CETI          0.2686  0.2793  0.1858  │     5.0%   5.7%   0.0%
  LSTM_BASE            0.2478  0.2549  0.1936  │     2.9%   3.3%   0.0%
  LSTM_CETI            0.2624  0.2780  0.1426  │     3.6%   4.1%   0.0%
  LSTM_CETI_2S         0.2439  0.2504  0.1945  │     2.9%   3.3%   0.0%


## Cell 11, Figures


In [31]:
COLORS = {'CAViaR_BASE':'#2166ac','CAViaR_CETI':'#92c5de',
          'LSTM_BASE':'#d6604d','LSTM_CETI':'#f4a582',
          'LSTM_CETI_2S':'#7b3294','ret':'#555555'}

def shade(ax, idx):
    for s,e,_ in CRISIS_SHADING:
        s,e = pd.Timestamp(s), pd.Timestamp(e)
        if s<=idx[-1] and e>=idx[0]:
            ax.axvspan(max(s,idx[0]),min(e,idx[-1]),alpha=.12,color='#ffd966',zorder=0)

alpha = 0.05
for asset in ASSETS:
    d_te = DATASETS[asset][DATASETS[asset]['sample']=='test']
    y_te = d_te['y'].values * R_SCALE; idx = d_te.index

    # ── Fig 1: VaR trajectories ──────────────────────────────────────────
    fig, axes = plt.subplots(2,1,figsize=(16,8),sharex=True)
    for ax, fam, ms in zip(axes,['CAViaR','LSTM'],
                           [['CAViaR_BASE','CAViaR_CETI'],['LSTM_BASE','LSTM_CETI']]):
        ax.plot(idx, y_te, color=COLORS['ret'], alpha=.5, lw=.7, label='Return')
        for m in ms:
            v = FORECASTS[(m,asset,alpha)]['var']
            ax.plot(idx, v, color=COLORS[m], lw=1.5, label=m)
        v_c = FORECASTS[(ms[1],asset,alpha)]['var']
        vx  = idx[y_te < v_c]
        ax.scatter(vx, y_te[y_te<v_c], color='red', s=20, zorder=5, label='Violations (CETI)')
        shade(ax, idx); ax.axhline(0,color='grey',lw=.5,ls='--')
        ax.set_ylabel('Return (%)')
        ax.legend(fontsize=11,ncol=2,loc='lower left'); ax.grid(axis='y',alpha=.3)
    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    fp = _os.path.join(PLOTS_DIR, f'01_var_trajectories_{asset}.png')
    plt.savefig(fp,dpi=180,bbox_inches='tight'); plt.close(); print('Saved:',fp)

    # ── Fig 2: Pinball full vs crisis ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10,4))
    models = ['CAViaR_BASE','CAViaR_CETI','LSTM_BASE','LSTM_CETI','LSTM_CETI_2S']  # was CLRI
    sub = RESULTS_DF[(RESULTS_DF['Asset']==asset)&(RESULTS_DF['Alpha']==alpha)]
    sub = sub.set_index('Model')
    x = np.arange(len(models)); w=.35
    ax.bar(x-w/2,[sub.loc[m,'Pinball'] for m in models],w,label='Full',color='#4292c6',alpha=.85)
    ax.bar(x+w/2,[sub.loc[m,'Pinball_crisis'] for m in models],w,label='Crisis',color='#ef6548',alpha=.85)
    ax.set_xticks(x); ax.set_xticklabels(models,rotation=15,ha='right')
    ax.set_ylabel('Mean pinball loss (%)'); ax.legend(); ax.grid(axis='y',alpha=.3)
    plt.tight_layout()
    fp = _os.path.join(PLOTS_DIR, f'02_Q-Loss_{asset}.png')
    plt.savefig(fp,dpi=180,bbox_inches='tight'); plt.close(); print('Saved:',fp)

    # ── Fig 3: CETI vs VaR spread ────────────────────────────────────────
    ceti = DATASETS[asset][DATASETS[asset]['sample']=='test']['ceti_lag'].values  # was clri_lag, clri
    fig, axes = plt.subplots(3,1,figsize=(16,9),sharex=True)
    axes[0].plot(idx,ceti,color='#1a6b3c',lw=1.2)                                # was clri
    axes[0].axhline(0,color='grey',lw=.5,ls='--')
    axes[0].fill_between(idx,ceti,0,where=ceti<0,color='#c00000',alpha=.2)       # was clri
    shade(axes[0],idx); axes[0].set_ylabel('CETI (β̂ᴴ W104)')
    for ax, fam, base_m, ceti_m in zip(axes[1:],['CAViaR','LSTM'],             # was clri_m
                                        ['CAViaR_BASE','LSTM_BASE'],
                                        ['CAViaR_CETI','LSTM_CETI']):           # was CLRI
        sp = FORECASTS[(ceti_m,asset,alpha)]['var'] - FORECASTS[(base_m,asset,alpha)]['var']  # was clri_m
        ax.plot(idx,sp,color=COLORS[ceti_m],lw=1.2)                            # was clri_m
        ax.axhline(0,color='grey',lw=.5,ls='--')
        ax.fill_between(idx,sp,0,where=sp<0,color=COLORS[ceti_m],alpha=.3)     # was clri_m
        shade(ax,idx); ax.set_ylabel('ΔVaR (%)')
    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    fp = _os.path.join(PLOTS_DIR, f'03_ceti_response_{asset}.png')             # was clri
    plt.savefig(fp,dpi=180,bbox_inches='tight'); plt.close(); print('Saved:',fp)

Saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_HORSERACE_V2/PLOTS/01_var_trajectories_BBVA.png
Saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_HORSERACE_V2/PLOTS/02_pinball_BBVA.png
Saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_HORSERACE_V2/PLOTS/03_ceti_response_BBVA.png
Saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_HORSERACE_V2/PLOTS/01_var_trajectories_BCP.png
Saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_HORSERACE_V2/PLOTS/02_pinball_BCP.png
Saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_HORSERACE_V2/PLOTS/03_ceti_response_BCP.png


## Cell 12, Final Conclusion


In [32]:
sep('HORSE RACE v2, FINAL SUMMARY')

print('\n== 1. DM Tests: CETI incremental value ==')
for asset in ASSETS:
    print(f'  {asset}:')
    for fam in ['CAViaR','LSTM','LSTM_2S']:
        row = DM_DF[(DM_DF['Family']==fam)&(DM_DF['Asset']==asset)&(DM_DF['Alpha']==.05)]
        if len(row): r=row.iloc[0]; print(f'    {fam:8s}: DM={r.DM_stat:+.3f} p={r.p_value:.3f} → {r.Conclusion}')

print('\n══ 2. Pinball improvement (τ=5%) ══')
for asset in ASSETS:
    sub = RESULTS_DF[(RESULTS_DF['Asset']==asset)&(RESULTS_DF['Alpha']==.05)].set_index('Model')['Pinball']
    print(f'  {asset}:')
    for bm,cm in [('CAViaR_BASE','CAViaR_CETI'),('LSTM_BASE','LSTM_CETI'),('LSTM_BASE','LSTM_CETI_2S')]:
        g=(sub[bm]-sub[cm])/sub[bm]*100
        print(f'    {bm}→{cm}: {g:+.2f}% {"↓ better" if sub[cm]<sub[bm] else "↑ worse"}')

print('\n══ 3. Coverage summary (τ=5%) ══')
for asset in ASSETS:
    sub = RESULTS_DF[(RESULTS_DF['Asset']==asset)&(RESULTS_DF['Alpha']==.05)]
    print(f'  {asset}:')
    for _,row in sub.iterrows():
        kp=row.Kupiec_p; cc=row.CC_p
        uc='✓' if (pd.notna(kp) and kp>.05) else '✗'
        co='✓' if (pd.notna(cc) and cc>.05) else '✗'
        print(f'    {row.Model:<18}: viol={row.ViolRate:.1%}  UC {uc}(p={kp})  CC {co}(p={cc})')

print(f'\nAll outputs → {OUTPUT_DIR}')



════════════════════════════════════════════════════════════
 HORSE RACE v2, FINAL SUMMARY
════════════════════════════════════════════════════════════

== 1. DM Tests: CETI incremental value ==
  BBVA:
    CAViaR  : DM=+0.811 p=0.418 → ns
    LSTM    : DM=-0.182 p=0.855 → ns
    LSTM_2S : DM=+2.965 p=0.003 → CETI better
  BCP:
    CAViaR  : DM=+0.755 p=0.450 → ns
    LSTM    : DM=-0.978 p=0.328 → ns
    LSTM_2S : DM=+1.129 p=0.259 → ns

══ 2. Pinball improvement (τ=5%) ══
  BBVA:
    CAViaR_BASE→CAViaR_CETI: +0.48% ↓ better
    LSTM_BASE→LSTM_CETI: -0.68% ↑ worse
    LSTM_BASE→LSTM_CETI_2S: +3.58% ↓ better
  BCP:
    CAViaR_BASE→CAViaR_CETI: +2.65% ↓ better
    LSTM_BASE→LSTM_CETI: -5.90% ↑ worse
    LSTM_BASE→LSTM_CETI_2S: +1.56% ↓ better

══ 3. Coverage summary (τ=5%) ══
  BBVA:
    CAViaR_BASE       : viol=1.4%  UC ✗(p=0.0239)  CC ✓(p=0.0758)
    CAViaR_CETI       : viol=1.4%  UC ✗(p=0.0239)  CC ✓(p=0.0758)
    LSTM_BASE         : viol=2.9%  UC ✓(p=0.2138)  CC ✓(p=0.1038)
    LSTM